# Helmet / No-Helmet / Rider Boosted Fine-Tune

Notebook nay tao dataset 3 class tu bo 5 class, uu tien cai thien class `no_helmet`, sau do fine-tune YOLOv8 tren Kaggle.

Diem nhan chinh:
- remap 5 class -> 3 class
- giu nguyen `valid` neu dataset da co san
- oversample anh train co chua `no_helmet`
- train voi `imgsz` lon hon de bat vat the nho tot hon


In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
import ast
import random
import shutil
from collections import Counter

from PIL import Image, ImageDraw
from IPython.display import display
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
def parse_data_yaml(path: Path):
    info = {}
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or ':' not in line:
            continue
        key, value = line.split(':', 1)
        key = key.strip()
        value = value.strip()
        if key in {'train', 'val', 'test'}:
            info[key] = value
        elif key == 'nc':
            info[key] = int(value)
        elif key == 'names':
            info[key] = ast.literal_eval(value)
    return info


def find_data_yaml(search_root=Path('/kaggle/input')):
    candidates = sorted(search_root.rglob('data.yaml'), key=lambda p: len(str(p)))
    if not candidates:
        raise FileNotFoundError('Khong tim thay data.yaml trong /kaggle/input')
    return candidates[0]


def detect_existing_split(root: Path, names):
    for split_name in names:
        image_dir = root / split_name / 'images'
        label_dir = root / split_name / 'labels'
        if image_dir.exists() and label_dir.exists():
            return split_name
    return None


def load_labels(label_path: Path):
    labels = []
    for raw_line in label_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line:
            continue
        class_id, x, y, w, h = line.split()
        labels.append((int(class_id), float(x), float(y), float(w), float(h)))
    return labels


def remap_label_lines(label_path: Path, class_map):
    remapped = []
    for class_id, x, y, w, h in load_labels(label_path):
        if class_id not in class_map:
            continue
        remapped.append(f"{class_map[class_id]} {x} {y} {w} {h}")
    return remapped


def summarize_split(split_name, root, class_names):
    image_dir = root / split_name / 'images'
    label_dir = root / split_name / 'labels'
    images = sorted([p for p in image_dir.glob('*') if p.is_file()])
    labels = sorted(label_dir.glob('*.txt')) if label_dir.exists() else []
    counts = Counter()
    total_boxes = 0
    for label_path in labels:
        for class_id, *_ in load_labels(label_path):
            counts[class_id] += 1
            total_boxes += 1
    print(f'[{split_name}] images={len(images)} labels={len(labels)} total_boxes={total_boxes}')
    for class_id in sorted(counts):
        print(f'  class {class_id} ({class_names[class_id]}): {counts[class_id]}')
    return images, label_dir


def draw_sample(image_path: Path, label_dir: Path, class_names):
    colors = ['lime', 'red', 'cyan']
    label_path = label_dir / f'{image_path.stem}.txt'
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    w, h = image.size
    if label_path.exists():
        for class_id, x, y, bw, bh in load_labels(label_path):
            x1 = (x - bw / 2) * w
            y1 = (y - bh / 2) * h
            x2 = (x + bw / 2) * w
            y2 = (y + bh / 2) * h
            color = colors[class_id % len(colors)]
            draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
            draw.text((x1, max(0, y1 - 12)), class_names[class_id], fill=color)
    return image


In [ ]:
data_yaml_input = find_data_yaml()
dataset_root_input = data_yaml_input.parent
dataset_info = parse_data_yaml(data_yaml_input)

print('Found data.yaml:', data_yaml_input)
print('Dataset root:', dataset_root_input)
print(data_yaml_input.read_text())

source_train_split = detect_existing_split(dataset_root_input, ['train'])
source_valid_split = detect_existing_split(dataset_root_input, ['valid', 'val'])
source_test_split = detect_existing_split(dataset_root_input, ['test'])

print('source_train_split =', source_train_split)
print('source_valid_split =', source_valid_split)
print('source_test_split =', source_test_split)

In [ ]:
OLD_HELMET_ID = 2
OLD_NO_HELMET_ID = 1
OLD_RIDER_ID = 4
CLASS_MAP = {
    OLD_HELMET_ID: 0,
    OLD_NO_HELMET_ID: 1,
    OLD_RIDER_ID: 2,
}
CLASS_NAMES = ['helmet', 'no_helmet', 'rider']

OVERSAMPLE_NO_HELMET = True
NO_HELMET_CLASS_ID = 1
TARGET_NO_HELMET_BOX_RATIO = 0.95
MAX_EXTRA_ROUNDS = 2
VALID_FRACTION_IF_MISSING = 0.15

reduced_root = Path('/kaggle/working/helmet_dataset_3class_boosted')
if reduced_root.exists():
    shutil.rmtree(reduced_root)

for split in ['train', 'valid', 'test']:
    (reduced_root / split / 'images').mkdir(parents=True, exist_ok=True)
    (reduced_root / split / 'labels').mkdir(parents=True, exist_ok=True)

def copy_entry(image_path: Path, label_lines, dst_split: str):
    dst_image = reduced_root / dst_split / 'images' / image_path.name
    dst_label = reduced_root / dst_split / 'labels' / f'{image_path.stem}.txt'
    shutil.copy2(image_path, dst_image)
    dst_label.write_text('\n'.join(label_lines) + '\n', encoding='utf-8')


train_candidates = []
source_train_dir = dataset_root_input / source_train_split / 'images'
source_train_label_dir = dataset_root_input / source_train_split / 'labels'
for image_path in sorted(source_train_dir.glob('*')):
    if not image_path.is_file():
        continue
    label_path = source_train_label_dir / f'{image_path.stem}.txt'
    if not label_path.exists():
        continue
    remapped = remap_label_lines(label_path, CLASS_MAP)
    if remapped:
        train_candidates.append((image_path, remapped))

print('train candidates with kept classes =', len(train_candidates))

if source_valid_split is not None:
    for image_path, remapped in train_candidates:
        copy_entry(image_path, remapped, 'train')

    source_valid_dir = dataset_root_input / source_valid_split / 'images'
    source_valid_label_dir = dataset_root_input / source_valid_split / 'labels'
    valid_kept = 0
    for image_path in sorted(source_valid_dir.glob('*')):
        if not image_path.is_file():
            continue
        label_path = source_valid_label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            continue
        remapped = remap_label_lines(label_path, CLASS_MAP)
        if not remapped:
            continue
        copy_entry(image_path, remapped, 'valid')
        valid_kept += 1
    print('valid kept =', valid_kept)
else:
    shuffled = train_candidates[:]
    random.shuffle(shuffled)
    valid_count = max(1, int(len(shuffled) * VALID_FRACTION_IF_MISSING))
    valid_lookup = {str(image_path) for image_path, _ in shuffled[:valid_count]}
    for image_path, remapped in shuffled:
        dst_split = 'valid' if str(image_path) in valid_lookup else 'train'
        copy_entry(image_path, remapped, dst_split)
    print('valid created from train =', valid_count)

if source_test_split is not None:
    source_test_dir = dataset_root_input / source_test_split / 'images'
    source_test_label_dir = dataset_root_input / source_test_split / 'labels'
    test_kept = 0
    for image_path in sorted(source_test_dir.glob('*')):
        if not image_path.is_file():
            continue
        label_path = source_test_label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            continue
        remapped = remap_label_lines(label_path, CLASS_MAP)
        if not remapped:
            continue
        copy_entry(image_path, remapped, 'test')
        test_kept += 1
    print('test kept =', test_kept)

yaml_lines = ['train: train/images', 'val: valid/images']
if any((reduced_root / 'test' / 'images').glob('*')):
    yaml_lines.append('test: test/images')
yaml_lines += ['', 'nc: 3', f'names: {CLASS_NAMES}', '']
(reduced_root / 'data.yaml').write_text('\n'.join(yaml_lines), encoding='utf-8')

print((reduced_root / 'data.yaml').read_text())

In [ ]:
if OVERSAMPLE_NO_HELMET:
    image_dir = reduced_root / 'train' / 'images'
    label_dir = reduced_root / 'train' / 'labels'
    class_box_counts = Counter()
    no_helmet_samples = []
    for image_path in sorted(image_dir.glob('*')):
        if not image_path.is_file() or '_os' in image_path.stem:
            continue
        label_path = label_dir / f'{image_path.stem}.txt'
        if not label_path.exists():
            continue
        labels = load_labels(label_path)
        no_helmet_boxes = 0
        for class_id, *_ in labels:
            class_box_counts[class_id] += 1
            if class_id == NO_HELMET_CLASS_ID:
                no_helmet_boxes += 1
        if no_helmet_boxes > 0:
            no_helmet_samples.append((image_path, label_path, no_helmet_boxes))

    helmet_boxes = class_box_counts.get(0, 0)
    current_no_helmet_boxes = class_box_counts.get(NO_HELMET_CLASS_ID, 0)
    target_no_helmet_boxes = int(helmet_boxes * TARGET_NO_HELMET_BOX_RATIO)
    needed_boxes = max(0, target_no_helmet_boxes - current_no_helmet_boxes)

    created = 0
    created_boxes = 0
    if needed_boxes > 0 and no_helmet_samples:
        sample_pool = no_helmet_samples[:]
        random.shuffle(sample_pool)
        round_idx = 0
        while needed_boxes > 0 and round_idx < MAX_EXTRA_ROUNDS:
            for image_path, label_path, no_helmet_boxes in sample_pool:
                new_image_path = image_dir / f'{image_path.stem}_osr{round_idx + 1}_{created + 1}{image_path.suffix}'
                new_label_path = label_dir / f'{image_path.stem}_osr{round_idx + 1}_{created + 1}.txt'
                shutil.copy2(image_path, new_image_path)
                shutil.copy2(label_path, new_label_path)
                created += 1
                created_boxes += no_helmet_boxes
                needed_boxes -= no_helmet_boxes
                if needed_boxes <= 0:
                    break
            round_idx += 1

    print('Oversample summary:')
    print(' helmet boxes =', helmet_boxes)
    print(' current no_helmet boxes =', current_no_helmet_boxes)
    print(' target no_helmet boxes =', target_no_helmet_boxes)
    print(' created extra samples =', created)
    print(' created no_helmet boxes ~= ', created_boxes)
else:
    print('Oversampling disabled.')

In [ ]:
train_images, train_label_dir = summarize_split('train', reduced_root, CLASS_NAMES)
valid_images, valid_label_dir = summarize_split('valid', reduced_root, CLASS_NAMES)
if any((reduced_root / 'test' / 'images').glob('*')):
    test_images, test_label_dir = summarize_split('test', reduced_root, CLASS_NAMES)

In [ ]:
sample_paths = random.sample(train_images, k=min(4, len(train_images)))
for sample_path in sample_paths:
    print(sample_path.name)
    display(draw_sample(sample_path, train_label_dir, CLASS_NAMES))

In [ ]:
AUTO_FIND_CUSTOM_CHECKPOINT = True
CUSTOM_CHECKPOINT_FILENAME = 'best.pt'
FALLBACK_MODEL = 'yolov8l.pt'
STAGE1_EPOCHS = 15
STAGE1_IMGSZ = 768
STAGE1_BATCH = 8
STAGE1_FREEZE = 10
STAGE2_EPOCHS = 80
STAGE2_IMGSZ = 896
STAGE2_BATCH = 4
PROJECT = '/kaggle/working/runs/helmet_3class_project'
RUN_NAME = 'push90_two_stage'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

def find_custom_checkpoint(search_root=Path('/kaggle/input'), filename='best.pt'):
    candidates = sorted(search_root.rglob(filename), key=lambda p: (len(str(p)), str(p)))
    if not candidates:
        return None, []
    return candidates[0], candidates

custom_checkpoint, checkpoint_candidates = (None, [])
if AUTO_FIND_CUSTOM_CHECKPOINT:
    custom_checkpoint, checkpoint_candidates = find_custom_checkpoint(filename=CUSTOM_CHECKPOINT_FILENAME)
    print('Checkpoint candidates found:')
    if checkpoint_candidates:
        for path in checkpoint_candidates[:10]:
            print(' -', path)
    else:
        print(' - none')

MODEL_NAME = str(custom_checkpoint) if custom_checkpoint is not None else FALLBACK_MODEL
print('Using pretrained checkpoint:', MODEL_NAME)

# Cau hinh nay uu tien day do chinh xac len muc cao nhat co the tren Kaggle T4.
# Stage 1: freeze mot phan backbone, hoc nhe de on dinh checkpoint.
# Stage 2: mo full model, anh lon hon, LR nho hon de fine-tune sat du lieu.

stage1_name = RUN_NAME + '_stage1'
stage2_name = RUN_NAME + '_stage2'

stage1_model = YOLO(MODEL_NAME)
stage1_results = stage1_model.train(
    data=str(reduced_root / 'data.yaml'),
    epochs=STAGE1_EPOCHS,
    imgsz=STAGE1_IMGSZ,
    batch=STAGE1_BATCH,
    device=DEVICE,
    project=PROJECT,
    name=stage1_name,
    pretrained=True,
    optimizer='AdamW',
    freeze=STAGE1_FREEZE,
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=10,
    workers=2,
    cache=True,
    cos_lr=True,
    close_mosaic=5,
    degrees=0.0,
    perspective=0.0,
    translate=0.05,
    scale=0.20,
    fliplr=0.5,
    mosaic=0.2,
    mixup=0.0,
    copy_paste=0.0,
    seed=SEED,
)

stage1_best = Path(PROJECT) / stage1_name / 'weights' / 'best.pt'
print('Stage 1 save dir:', stage1_results.save_dir)
print('Stage 1 best checkpoint:', stage1_best)

stage2_model = YOLO(str(stage1_best))
stage2_results = stage2_model.train(
    data=str(reduced_root / 'data.yaml'),
    epochs=STAGE2_EPOCHS,
    imgsz=STAGE2_IMGSZ,
    batch=STAGE2_BATCH,
    device=DEVICE,
    project=PROJECT,
    name=stage2_name,
    pretrained=True,
    optimizer='AdamW',
    lr0=0.0005,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=20,
    workers=2,
    cache=True,
    cos_lr=True,
    close_mosaic=8,
    degrees=0.0,
    perspective=0.0,
    translate=0.04,
    scale=0.20,
    fliplr=0.5,
    mosaic=0.15,
    mixup=0.0,
    copy_paste=0.0,
    seed=SEED,
)

FINAL_RUN_NAME = stage2_name
FINAL_IMGSZ = STAGE2_IMGSZ
print('Stage 2 save dir:', stage2_results.save_dir)

In [ ]:
best_model_path = Path(PROJECT) / FINAL_RUN_NAME / 'weights' / 'best.pt'
best_model = YOLO(str(best_model_path))
print('Validating standard best model:', best_model_path)
metrics = best_model.val(data=str(reduced_root / 'data.yaml'), imgsz=FINAL_IMGSZ, device=DEVICE)
print('Running TTA validation...')
metrics_tta = best_model.val(
    data=str(reduced_root / 'data.yaml'),
    imgsz=FINAL_IMGSZ,
    device=DEVICE,
    augment=True,
    name='val_tta'
)
metrics_tta

In [ ]:
predict_source = str(valid_images[0]) if valid_images else str(train_images[0])
predictions = best_model.predict(source=predict_source, imgsz=FINAL_IMGSZ, conf=0.25, save=True)
predictions[0].save_dir